# Well Played, Mauer: Calculating wOBA

## Section 1: Calculating wOBA

In [1]:
# import the necessary packages
import os
import sys
import pandas as pd

In [17]:
# set up the file paths
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
raw_data_dir = os.path.join(project_root, 'data', 'raw')
processed_data_dir = os.path.join(project_root, 'data', 'processed')

# test the paths
# print(f'Project Root: {project_root}')
# print(f'Raw Data Directory: {raw_data_dir}')
# print(f'Processed Data Directory: {processed_data_dir}')

Project Root: /Users/ajgafford/Documents/Projects/Reached_on_Error/Well_Played_Mauer
Raw Data Directory: /Users/ajgafford/Documents/Projects/Reached_on_Error/Well_Played_Mauer/data/raw
Processed Data Directory: /Users/ajgafford/Documents/Projects/Reached_on_Error/Well_Played_Mauer/data/processed


In [3]:
# read in the csv for qualified batters in 2009
# data courtesy of stathead
filename = 'mlb_qualified_batters_2009.csv'
csv_path = os.path.join(raw_data_dir, filename)
batters = pd.read_csv(csv_path)
batters.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 41 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Rk                 155 non-null    int64  
 1   Player             155 non-null    object 
 2   Age                155 non-null    int64  
 3   Team               155 non-null    object 
 4   Lg                 155 non-null    object 
 5   G                  155 non-null    int64  
 6   PA                 155 non-null    int64  
 7   AB                 155 non-null    int64  
 8   R                  155 non-null    int64  
 9   H                  155 non-null    int64  
 10  1B                 155 non-null    int64  
 11  2B                 155 non-null    int64  
 12  3B                 155 non-null    int64  
 13  HR                 155 non-null    int64  
 14  RBI                155 non-null    int64  
 15  SB                 155 non-null    int64  
 16  CS                 155 non

Using Stathead's Batting Event Finder, I found the RE24 and total number of each event for the following:

- **Unintentional Walks (uBB)**
- **Hits By Pitch (HBP)**
- **Singles (1B)**
- **Doubles (2B)**
- **Triples (3B)**
- **Home Runs (HR)**
- **Outs**

These are batter events that batters have the most control over.

In [4]:
# define the run value for each event
run_values = [
    4941 / 15441, # uBB
    539 / 1590, # HBP
    13457 / 28796, #1B
    6660 / 8737, # 2B
    982 / 949, # 3B
    6994 / 5042, # HR
    -34911 / 123241 # Outs
]

In [5]:
for rv in run_values:
    print(f'{rv:.3f}')

0.320
0.339
0.467
0.762
1.035
1.387
-0.283


We could stop there, but it might make more sense to compare these values to how much they contribute relative to an **out**, which is a zero in terms of other rate metrics.

In [6]:
for i in range(7):
    run_values[i] -= run_values[6]

In [7]:
for rv in run_values:
    print(f'{rv:.3f}')

0.603
0.622
0.751
1.046
1.318
1.670
0.000


Again, we could stop there, but it might be more useful to scale this calculation to something familiar. Tom Tango, the creator of **weighted On-Base Average (wOBA)**, decided to scale it to **OBP**.

In [8]:
# calculate the league unscaled woba
lg_unscaled_woba = (run_values[0] * 15441 + run_values[1] * 1590 + run_values[2] * 28796 + run_values[3] * 8737 + run_values[4] * 949 + run_values[5] * 5042) / (165849 + 16620 - 1179 + 1590 + 1366)
print(f'League Unscaled wOBA: {lg_unscaled_woba:.3f}')

League Unscaled wOBA: 0.275


In [9]:
# calculate the league obp sans ibb
lg_obp_sans_ibb = (43524 + 16620 - 1179 + 1590) / (165849 + 16620 - 1179 + 1590 + 1366)
print(f'League OBP w/o IBB: {lg_obp_sans_ibb:.3f}')

League OBP w/o IBB: 0.329


In [10]:
# find the constant to scale woba
woba_scale = lg_obp_sans_ibb / lg_unscaled_woba
print(f'wOBA Scale: {woba_scale:.3f}')

wOBA Scale: 1.194


In [11]:
for i in range(7):
    run_values[i] *= woba_scale

for rv in run_values:
    print(f'{rv:.3f}')

0.720
0.743
0.896
1.248
1.573
1.994
0.000


In [12]:
# extract unintentional walks
batters['UBB'] = batters['BB'] - batters['IBB']

In [13]:
# define the wOBA denominator
batters['wOBA_denom'] = batters['AB'] + batters['UBB'] + batters['SF'] + batters['HBP']

In [14]:
# calculate wOBA
batters['wOBA'] = (run_values[0] * batters['UBB'] + run_values[1] * batters['HBP'] + run_values[2] * batters['1B'] + run_values[3] * batters['2B'] + run_values[4] * batters['3B'] + run_values[5] * batters['HR']) / batters['wOBA_denom']

In [15]:
# export the processed dataframe to a csv
filename = 'mlb_qualified_batters_2009_wOBA.csv'
csv_path = os.path.join(processed_data_dir, filename)
batters[['Player', 'Player-additional', 'wOBA']].to_csv(csv_path)

## Section 2: Regression Modeling with wOBA

In a previous project, The Figgins-Hill Convergence, I used regression modeling with the **Triple Slash Line** and **OPS** to see how well each stat predicted a player's **RE24**, or net run expectancy for a season. Now, I'd like to do the same with **wOBA**. My dataset does not directly contain **wOBA** or anything similar, but I can use the components of **wOBA** plus yearly constants from Fangraphs to create it myself.

In [18]:
# read in the csv for qualified batters from 2006 to 2015
# data courtesy of stathead
filename = 'mlb_qualified_batters_2006_2015_processed.csv'
csv_path = os.path.join(processed_data_dir, filename)
batters = pd.read_csv(csv_path)
batters.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1495 entries, 0 to 1494
Data columns (total 52 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         1495 non-null   int64  
 1   Rk                 1495 non-null   int64  
 2   Player             1495 non-null   object 
 3   Season             1495 non-null   int64  
 4   Age                1495 non-null   int64  
 5   Team               1495 non-null   object 
 6   Lg                 1495 non-null   object 
 7   G                  1495 non-null   int64  
 8   PA                 1495 non-null   int64  
 9   AB                 1495 non-null   int64  
 10  R                  1495 non-null   int64  
 11  H                  1495 non-null   int64  
 12  1B                 1495 non-null   int64  
 13  2B                 1495 non-null   int64  
 14  3B                 1495 non-null   int64  
 15  HR                 1495 non-null   int64  
 16  RBI                1495 

In [19]:
# extract unintentional walks
batters['UBB'] = batters['BB'] - batters['IBB']

In [26]:
# create a dataframe of wOBA coefficients
# data courtesy of Fangraphs
data = [
    [2006, .708, .737, .890, 1.241, 1.557, 1.970],
    [2007, .711, .741, .896, 1.253, 1.575, 1.999],
    [2008, .708, .739, .896, 1.259, 1.587, 2.024],
    [2009, .707, .737, .895, 1.258, 1.585, 2.023],
    [2010, .701, .732, .895, 1.270, 1.608, 2.072],
    [2011, .694, .726, .890, 1.270, 1.611, 2.086],
    [2012, .691, .722, .884, 1.257, 1.593, 2.058],
    [2013, .690, .722, .888, 1.271, 1.616, 2.101],
    [2014, .689, .722, .892, 1.283, 1.635, 2.135],
    [2015, .687, .718, .881, 1.256, 1.594, 2.065],
]

cols = ['Season', 'UBB', 'HBP', '1B', '2B', '3B', 'HR']
weights_df = pd.DataFrame(data, columns=cols)

woba_weights = (
    weights_df
    .set_index('Season')
    .to_dict(orient='index')
)

In [29]:
# define a function that calculates wOBA by season
def calculate_woba(row, weights):
    season = row['Season']
    w = weights[season]

    numerator = (
        w['UBB'] * row['UBB'] +
        w['HBP'] * row['HBP'] +
        w['1B']  * row['1B'] +
        w['2B']  * row['2B'] +
        w['3B']  * row['3B'] +
        w['HR']  * row['HR']
    )

    denominator = (
        row['AB'] +
        row['UBB'] +
        row['HBP'] +
        row['SF']
    )

    return numerator / denominator

In [30]:
batters['wOBA'] = batters.apply(calculate_woba, axis=1, weights=woba_weights)

In [33]:
batters[['Season', 'Player', 'wOBA']].sort_values(by='wOBA', ascending=False).head(10)

,Season,Player,wOBA
3,2015,Bryce Harper,0.461183
5,2008,Albert Pujols,0.458839
2,2013,Miguel Cabrera,0.455043
29,2006,Travis Hafner,0.449980
20,2007,David Ortiz,0.449272
1,2009,Albert Pujols,0.447166
7,2006,Albert Pujols,0.447129
70,2008,Chipper Jones,0.445386
37,2010,Josh Hamilton,0.445258
0,2007,Álex Rodríguez,0.444594


With **wOBA** calculated for these batters, I can now run regression analysis.

In [38]:
import statsmodels.api as sm
import numpy as np

In [41]:
# define the variables
X = batters['wOBA']
X = sm.add_constant(X)
y = batters['RE24']

# create the regression model
model = sm.OLS(y, X).fit()

# extract the parameters\n',
intercept = model.params['const']
slope = model.params['wOBA']
r2 = model.rsquared
r = np.sign(slope) * np.sqrt(r2)

# print clean output
print('wOBA Model')
print(f'Equation: RE24 = {intercept:.4f} + {slope:.4f}·wOBA')
print(f'R²: {r2:.4f}')
print(f'r:  {r:.4f}')

wOBA Model
Equation: RE24 = -158.8760 + 500.1666·wOBA
R²: 0.8057
r:  0.8976


In [44]:
# find the predicted RE24
batters['pRE24_wOBA'] = -158.876 + 500.1666 * batters['wOBA']

# find the residual
batters['rRE24_wOBA'] = batters['RE24'] - batters['pRE24_wOBA']

In [45]:
# export the processed dataframe to a csv
filename = 'mlb_qualified_batters_2006_2015_wOBA.csv'
csv_path = os.path.join(processed_data_dir, filename)
batters[['Player', 'Player-additional', 'Season', 'wOBA', 'rRE24_wOBA']].to_csv(csv_path)